# Análise de Consumo de Eletricidade

Este notebook permite analisar dados de consumo de eletricidade de múltiplos meses.

## Passos para configurar o ambiente:

1. **Criar e ativar o ambiente virtual:**
   ```bash
   python3 -m venv venv
   source venv/bin/activate  # No Linux/Mac
   # ou
   venv\Scripts\activate  # No Windows
   ```

2. **Instalar as dependências:**
   ```bash
   pip install -r requirements.txt
   ```

3. **Iniciar o Jupyter:**
   ```bash
   jupyter notebook
   ```

## Estrutura de Pastas

- `data/` - Ficheiros CSV originais (um por mês)
- `processed/` - Dados processados e exportados

## Como Adicionar Novos Meses

Basta colocar novos ficheiros CSV na pasta `data/` com o formato `consumo_NOME_MES.csv`.
O notebook detectará automaticamente todos os ficheiros e carregará os dados.

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
import glob
from pathlib import Path

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Bibliotecas importadas com sucesso!")

## 2. Carregamento Automático de Todos os Ficheiros CSV

In [ ]:
# Definir caminhos
DATA_DIR = 'data'
PROCESSED_DIR = 'processed'

# Criar pasta processada se não existir
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Encontrar todos os ficheiros CSV na pasta data/
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

print(f"Ficheiros CSV encontrados em '{DATA_DIR}/': {len(csv_files)}")
for i, file in enumerate(csv_files, 1):
    print(f"  {i}. {os.path.basename(file)}")

In [ ]:
# Função para carregar e processar um ficheiro CSV
def carregar_csv(caminho_ficheiro):
    """Carrega um ficheiro CSV e retorna um DataFrame processado."""
    nome_ficheiro = os.path.basename(caminho_ficheiro)
    
    # Carregar o ficheiro
    df = pd.read_csv(caminho_ficheiro, encoding='utf-8-sig')
    
    # Adicionar coluna com o nome do ficheiro/mês
    df['Arquivo'] = nome_ficheiro
    
    # Converter colunas de data e hora para datetime
    df['Data'] = pd.to_datetime(df['Data'], format='%Y/%m/%d')
    df['Hora'] = pd.to_datetime(df['Hora'], format='%H:%M').dt.time
    
    # Criar coluna datetime combinada
    df['DataHora'] = df.apply(lambda row: datetime.combine(row['Data'], row['Hora']), axis=1)
    
    # Criar colunas adicionais para análise
    df['Dia'] = df['Data'].dt.day
    df['Mes'] = df['Data'].dt.month
    df['Ano'] = df['Data'].dt.year
    df['DiaSemana'] = df['Data'].dt.dayofweek
    df['NomeDiaSemana'] = df['Data'].dt.day_name()
    df['NomeMes'] = df['Data'].dt.month_name()
    df['HoraNum'] = df['DataHora'].dt.hour
    
    # Converter consumo para numérico
    df['Consumo registado (kW)'] = pd.to_numeric(df['Consumo registado (kW)'], errors='coerce')
    
    return df, nome_ficheiro

# Carregar todos os ficheiros
dfs = []
nomes_arquivos = []

for arquivo in csv_files:
    df, nome = carregar_csv(arquivo)
    dfs.append(df)
    nomes_arquivos.append(nome)
    print(f"✓ Carregado: {nome} ({len(df)} registros)")

# Combinar todos os DataFrames
df_completo = pd.concat(dfs, ignore_index=True)

print(f"\nTotal de registros combinados: {len(df_completo)}")

In [ ]:
# Informações sobre o dataset combinado
print("Informações do dataset combinado:")
df_completo.info()

print("\nEstatísticas descritivas:")
df_completo.describe()

In [ ]:
# Mostrar as primeiras linhas
print("Primeiras linhas do dataset combinado:")
df_completo.head(10)

## 3. Filtragem de Dados

In [ ]:
# Filtrar apenas dados reais (excluir estimados)
df_real = df_completo[df_completo['Estado'] == 'Real'].copy()
df_estimado = df_completo[df_completo['Estado'] == 'Estimado'].copy()

print(f"Total de registros: {len(df_completo)}")
print(f"Registros reais: {len(df_real)}")
print(f"Registros estimados: {len(df_estimado)}")

# Mostrar distribuição por arquivo/mês
print("\nDistribuição de registros por arquivo:")
print(df_completo['Arquivo'].value_counts().sort_index())

## 4. Análise Exploratória de Dados

In [ ]:
# Estatísticas básicas do consumo
print("Estatísticas do Consumo (kW) - Todos os meses:")
print(df_real['Consumo registado (kW)'].describe())

print(f"\nConsumo total: {df_real['Consumo registado (kW)'].sum():.2f} kW")
print(f"Consumo médio por registro: {df_real['Consumo registado (kW)'].mean():.4f} kW")
print(f"Consumo máximo: {df_real['Consumo registado (kW)'].max():.2f} kW")
print(f"Consumo mínimo: {df_real['Consumo registado (kW)'].min():.2f} kW")

In [ ]:
# Consumo por arquivo/mês
consumo_por_arquivo = df_real.groupby('Arquivo')['Consumo registado (kW)'].agg(['sum', 'mean', 'max', 'min', 'count']).round(4)
consumo_por_arquivo.columns = ['Total (kW)', 'Média (kW)', 'Máximo (kW)', 'Mínimo (kW)', 'Registros']
print("Consumo por arquivo/mês:")
consumo_por_arquivo

In [ ]:
# Consumo por mês
consumo_por_mes = df_real.groupby(['Ano', 'Mes', 'NomeMes'])['Consumo registado (kW)'].agg(['sum', 'mean', 'max', 'min']).round(4)
consumo_por_mes.columns = ['Total (kW)', 'Média (kW)', 'Máximo (kW)', 'Mínimo (kW)']
print("Consumo por mês:")
consumo_por_mes

In [ ]:
# Consumo por dia da semana
consumo_por_semana = df_real.groupby(['DiaSemana', 'NomeDiaSemana'])['Consumo registado (kW)'].sum().reset_index()
consumo_por_semana = consumo_por_semana.sort_values('DiaSemana')
print("Consumo por dia da semana:")
consumo_por_semana[['NomeDiaSemana', 'Consumo registado (kW)']]

In [ ]:
# Consumo por hora do dia
consumo_por_hora = df_real.groupby('HoraNum')['Consumo registado (kW)'].agg(['sum', 'mean', 'count']).round(4)
consumo_por_hora.columns = ['Total (kW)', 'Média (kW)', 'Contagem']
print("Consumo por hora do dia:")
consumo_por_hora

## 5. Visualizações

In [ ]:
# Gráfico 1: Consumo ao longo do tempo (todos os meses)
plt.figure(figsize=(20, 8))
plt.plot(df_real['DataHora'], df_real['Consumo registado (kW)'], linewidth=0.3, alpha=0.6)
plt.title('Consumo de Eletricidade ao Longo do Tempo - Todos os Meses', fontsize=14, fontweight='bold')
plt.xlabel('Data e Hora', fontsize=12)
plt.ylabel('Consumo (kW)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Consumo total por arquivo/mês
plt.figure(figsize=(12, 6))
consumo_por_arquivo['Total (kW)'].plot(kind='bar', color='steelblue')
plt.title('Consumo Total por Arquivo/Mês', fontsize=14, fontweight='bold')
plt.xlabel('Arquivo', fontsize=12)
plt.ylabel('Consumo Total (kW)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: Consumo por dia da semana
plt.figure(figsize=(10, 6))
sns.barplot(x='NomeDiaSemana', y='Consumo registado (kW)', data=consumo_por_semana)
plt.title('Consumo Total por Dia da Semana', fontsize=14, fontweight='bold')
plt.xlabel('Dia da Semana', fontsize=12)
plt.ylabel('Consumo Total (kW)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 4: Consumo médio por hora do dia
plt.figure(figsize=(12, 6))
consumo_por_hora['Média (kW)'].plot(kind='bar', color='coral')
plt.title('Consumo Médio por Hora do Dia', fontsize=14, fontweight='bold')
plt.xlabel('Hora do Dia', fontsize=12)
plt.ylabel('Consumo Médio (kW)', fontsize=12)
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 5: Boxplot do consumo por dia da semana
plt.figure(figsize=(12, 6))
sns.boxplot(x='NomeDiaSemana', y='Consumo registado (kW)', data=df_real)
plt.title('Distribuição do Consumo por Dia da Semana', fontsize=14, fontweight='bold')
plt.xlabel('Dia da Semana', fontsize=12)
plt.ylabel('Consumo (kW)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 6: Consumo por mês
plt.figure(figsize=(12, 6))
consumo_por_mes['Total (kW)'].plot(kind='bar', color='mediumseagreen')
plt.title('Consumo Total por Mês', fontsize=14, fontweight='bold')
plt.xlabel('Mês', fontsize=12)
plt.ylabel('Consumo Total (kW)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 7: Comparação de consumo médio por hora entre meses
if len(consumo_por_arquivo) > 1:
    plt.figure(figsize=(14, 8))
    
    for arquivo in nomes_arquivos:
        df_arquivo = df_real[df_real['Arquivo'] == arquivo]
        consumo_hora = df_arquivo.groupby('HoraNum')['Consumo registado (kW)'].mean()
        plt.plot(consumo_hora.index, consumo_hora.values, label=arquivo, linewidth=2)
    
    plt.title('Comparação de Consumo Médio por Hora - Todos os Meses', fontsize=14, fontweight='bold')
    plt.xlabel('Hora do Dia', fontsize=12)
    plt.ylabel('Consumo Médio (kW)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Apenas um mês de dados disponível. Adicione mais ficheiros CSV para comparação.")

## 6. Análise Avançada

In [ ]:
# Identificar picos de consumo
picos = df_real.nlargest(20, 'Consumo registado (kW)')[['DataHora', 'Consumo registado (kW)', 'Estado', 'Arquivo']]
print("Top 20 momentos de maior consumo:")
picos

In [ ]:
# Calcular custo estimado (assumindo uma tarifa média de €0.25/kWh)
tarifa = 0.25  # € por kWh

# Convertendo kW para kWh (cada registro é de 15 minutos = 0.25 horas)
df_real['Consumo_kWh'] = df_real['Consumo registado (kW)'] * 0.25
df_real['Custo_EUR'] = df_real['Consumo_kWh'] * tarifa

custo_total = df_real['Custo_EUR'].sum()
print(f"\nCusto total estimado: €{custo_total:.2f}")

# Custo por arquivo/mês
custo_por_arquivo = df_real.groupby('Arquivo')['Custo_EUR'].sum()
print("\nCusto por arquivo/mês:")
custo_por_arquivo

In [ ]:
# Análise de padrões de consumo
# Horas de pico (definidas como horas com consumo acima da média)
media_consumo = df_real['Consumo registado (kW)'].mean()
horas_pico = df_real[df_real['Consumo registado (kW)'] > media_consumo]['HoraNum'].value_counts().head(10)

print(f"Média de consumo: {media_consumo:.4f} kW")
print("\nHoras com mais frequência de consumo acima da média:")
horas_pico

In [ ]:
# Gráfico 8: Custo por arquivo/mês
plt.figure(figsize=(12, 6))
custo_por_arquivo.plot(kind='bar', color='green')
plt.title('Custo Estimado por Arquivo/Mês', fontsize=14, fontweight='bold')
plt.xlabel('Arquivo', fontsize=12)
plt.ylabel('Custo (€)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.axhline(y=custo_por_arquivo.mean(), color='red', linestyle='--', label=f'Média: €{custo_por_arquivo.mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 9: Tendência de consumo ao longo dos meses
if len(consumo_por_mes) > 1:
    plt.figure(figsize=(12, 6))
    plt.plot(range(len(consumo_por_mes)), consumo_por_mes['Total (kW)'], marker='o', linewidth=2, markersize=8)
    plt.title('Tendência de Consumo por Mês', fontsize=14, fontweight='bold')
    plt.xlabel('Mês (cronológico)', fontsize=12)
    plt.ylabel('Consumo Total (kW)', fontsize=12)
    plt.xticks(range(len(consumo_por_mes)), [f"{row['NomeMes']} {row['Ano']}" for _, row in consumo_por_mes.iterrows()], rotation=45, ha='right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Apenas um mês de dados disponível. Adicione mais ficheiros CSV para análise de tendência.")

## 7. Exportar Dados Processados

In [ ]:
# Exportar dados combinados processados
caminho_saida = os.path.join(PROCESSED_DIR, 'dados_combinados_processados.csv')
df_real.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
print(f"Dados combinados exportados para '{caminho_saida}'")

# Exportar resumo por arquivo/mês
caminho_resumo = os.path.join(PROCESSED_DIR, 'resumo_por_arquivo.csv')
consumo_por_arquivo.to_csv(caminho_resumo, encoding='utf-8-sig')
print(f"Resumo por arquivo exportado para '{caminho_resumo}'")

# Exportar resumo por mês
caminho_resumo_mes = os.path.join(PROCESSED_DIR, 'resumo_por_mes.csv')
consumo_por_mes.to_csv(caminho_resumo_mes, encoding='utf-8-sig')
print(f"Resumo por mês exportado para '{caminho_resumo_mes}'")

In [ ]:
# Exportar dados individuais processados para cada arquivo
for arquivo in nomes_arquivos:
    df_arquivo = df_real[df_real['Arquivo'] == arquivo].copy()
    
    # Criar nome de ficheiro de saída
    nome_saida = arquivo.replace('.csv', '_processado.csv')
    caminho_saida = os.path.join(PROCESSED_DIR, nome_saida)
    
    df_arquivo.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
    print(f"Dados de '{arquivo}' exportados para '{caminho_saida}'")

## 8. Resumo da Análise

In [ ]:
# Resumo final
print("="*70)
print("RESUMO DA ANÁLISE DE CONSUMO DE ELETRICIDADE")
print("="*70)
print(f"\nPeríodo analisado: {df_real['Data'].min().strftime('%d/%m/%Y')} a {df_real['Data'].max().strftime('%d/%m/%Y')}")
print(f"Total de meses analisados: {len(consumo_por_mes)}")
print(f"Total de registros: {len(df_real)}")
print(f"\nConsumo Total: {df_real['Consumo registado (kW)'].sum():.2f} kW")
print(f"Consumo Médio: {df_real['Consumo registado (kW)'].mean():.4f} kW")
print(f"Consumo Máximo: {df_real['Consumo registado (kW)'].max():.2f} kW")
print(f"Consumo Mínimo: {df_real['Consumo registado (kW)'].min():.2f} kW")
print(f"\nCusto Total Estimado: €{custo_total:.2f}")

print("\n" + "-"*70)
print("CONSUMO POR MÊS:")
print("-"*70)
for idx, row in consumo_por_mes.iterrows():
    ano = idx[0]
    mes = idx[1]
    nome_mes = idx[2]
    total_kw = row['Total (kW)']
    custo = total_kw * 0.25
    print(f"{nome_mes} {ano}: {total_kw:.2f} kW (€{custo:.2f})")

print("\n" + "-"*70)
print("MÊS COM MAIOR CONSUMO:")
print("-"*70)
mes_max = consumo_por_mes['Total (kW)'].idxmax()
ano_max = mes_max[0]
nome_mes_max = mes_max[2]
total_max = consumo_por_mes.loc[mes_max, 'Total (kW)']
custo_max = total_max * 0.25
print(f"{nome_mes_max} {ano_max}: {total_max:.2f} kW (€{custo_max:.2f})")

print("\n" + "-"*70)
print("MÊS COM MENOR CONSUMO:")
print("-"*70)
mes_min = consumo_por_mes['Total (kW)'].idxmin()
ano_min = mes_min[0]
nome_mes_min = mes_min[2]
total_min = consumo_por_mes.loc[mes_min, 'Total (kW)']
custo_min = total_min * 0.25
print(f"{nome_mes_min} {ano_min}: {total_min:.2f} kW (€{custo_min:.2f})")

print("\n" + "-"*70)
print("DIA DA SEMANA COM MAIOR CONSUMO:")
print("-"*70)
dia_max = consumo_por_semana.loc[consumo_por_semana['Consumo registado (kW)'].idxmax(), 'NomeDiaSemana']
print(f"{dia_max}")

print("\n" + "-"*70)
print("DIA DA SEMANA COM MENOR CONSUMO:")
print("-"*70)
dia_min = consumo_por_semana.loc[consumo_por_semana['Consumo registado (kW)'].idxmin(), 'NomeDiaSemana']
print(f"{dia_min}")

print("\n" + "="*70)